# Checkpoint 4: Interpreting p-values and Confidence Intervals

## 1. Loading df_clean

In [178]:
import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations

pd.set_option('display.max_columns', 40)

In [179]:
cols = [
    "id_x", "car_rel_url_x", "datetime_scrape", "price_x", "currency_x", "city",
    "production_year", "engine_displacement_num", "kilometrage_num", "Marka", "Model",
    "Sürətlər qutusu", "Vəziyyəti", "Ötürücü", "Ban növü", "views"
]

In [180]:
df = pd.read_csv("cars.csv", usecols = cols, parse_dates = ["datetime_scrape"])


In [181]:
df

,id_x,car_rel_url_x,datetime_scrape,price_x,currency_x,city,production_year,engine_displacement_num,kilometrage_num,views,Ban növü,Marka,Model,Sürətlər qutusu,Vəziyyəti,Ötürücü
0,3c234145-d57a-4ad6-9448-d43810fc3392,/autos/8748840-hyundai-i30,2024-09-13 20:32:19.751157+00,15000.0,AZN,bakı,2008,1.6,270000,492,"Hetçbek, 5 qapı",Hyundai,i30,Mexaniki,"Vuruğu yoxdur, rənglənməyib",Ön
1,c74ea36f-6be1-4de4-926d-e117197dcf00,/autos/8475807-lada-vaz-niva-travel,2024-09-13 20:32:19.751157+00,23700.0,AZN,bakı,2024,1.7,0,60189,"Offroader / SUV, 5 qapı",LADA (VAZ),Niva Travel,Mexaniki,"Vuruğu yoxdur, rənglənməyib",Tam
2,9cefceb0-024d-4581-a869-a3c2c68a9f95,/autos/8739686-toyota-land-cruiser,2024-09-13 20:32:19.751157+00,35600.0,$,bakı,2011,4.0,164750,2473,"Offroader / SUV, 5 qapı",Toyota,Land Cruiser,Avtomat,"Vuruğu yoxdur, rənglənməyib",Tam
3,459cc337-fb63-48de-9694-41554923d311,/autos/8712597-hyundai-elantra,2024-09-13 20:32:19.751157+00,26700.0,AZN,bakı,2018,2.0,126000,3727,Sedan,Hyundai,Elantra,Avtomat,"Vuruğu yoxdur, rənglənməyib",Ön
4,6c5ee8d8-1c6f-4fad-a694-957a4c43c25d,/autos/8674773-toyota-prius,2024-09-13 20:32:19.751157+00,10500.0,AZN,bakı,2007,1.5,354000,446,Liftbek,Toyota,Prius,Variator,"Vuruğu yoxdur, rənglənməyib",Ön
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
653716,16caa803-a546-455b-81ff-bc20868c2136,/autos/9081944-toyota-prius,2025-01-05 20:15:21.051803,10800.0,AZN,bakı,2008,1.5,320000,210,Liftbek,Toyota,Prius,Variator,"Vuruğu yoxdur, rənglənməyib",Ön
653717,2a26a6c1-8914-4b68-abb4-1fbd12ced7bc,/autos/9081939-uaz-hunter,2025-01-05 20:15:21.051803,9500.0,AZN,göygöl,2011,2.9,155000,1195,"Offroader / SUV, 5 qapı",UAZ,Hunter,Mexaniki,"Vuruğu yoxdur, rənglənməyib",Tam
653718,feb75615-0131-4d6f-b009-cc02faea4e01,/autos/9055034-hyundai-elantra,2025-01-05 20:15:21.051803,25400.0,AZN,bakı,2018,2.0,77926,1120,Sedan,Hyundai,Elantra,Avtomat,"Vuruğu yoxdur, rənglənməyib",Ön
653719,e5f8e957-5543-42be-b30f-78723ce551f9,/autos/9065903-jeep-grand-cherokee,2025-01-05 20:15:21.051803,10600.0,AZN,kürdəmir,1999,4.7,250000,1320,"Offroader / SUV, 5 qapı",Jeep,Grand Cherokee,Avtomat,"Vuruğu yoxdur, rənglənməyib",Tam


In [182]:
df_dedup = df.sort_values("datetime_scrape").drop_duplicates(subset = "car_rel_url_x", keep = "last").copy()

In [183]:
exchange_rate = {"AZN": 1.0, "$": 1.70, "€": 1.85}
df_dedup["price_azn"] = df_dedup["price_x"] * df_dedup["currency_x"].map(exchange_rate)

In [184]:
exclude_body_types = ["Yük maşını", "Motosiklet", "Avtobus", "Moped", "Kvadrosikl", "Dartqı", "Mikroavtobus"]
df_clean = df_dedup[~df_dedup["Ban növü"].isin(exclude_body_types)].copy()
df_clean = df_clean[df_clean["price_azn"] >= 1000].copy()

In [185]:
print("Cleaned dataset shape:", df_clean.shape)

Cleaned dataset shape: (149478, 17)


## 2. p-value

In [186]:
manual = df_clean.loc[df_clean["Sürətlər qutusu"] == "Mexaniki", "price_azn"]
auto = df_clean.loc[df_clean["Sürətlər qutusu"] == "Avtomat", "price_azn"]


In [187]:
t_stat, p_value = stats.ttest_ind(auto, manual, equal_var=False)  # Welch's t-test
print(f"Welch t-statistic: {t_stat:.2f}")
print(f"p-value: {p_value:.2e}")

Welch t-statistic: 166.72
p-value: 0.00e+00


If automatic and manual cars really had the same average price, seeing a gap this large between a sample of 37,161 manual cars and 98,161 automatic cars would be almost impossible. That's a strong enough reason to reject H0.

It doesn't say automatic cars cost 19,000 AZN more, it doesn't say having an automatic gearbox causes the higher price, and it doesn't tell me if the gap is actually big enough to matter. With a sample this large (135,000+ cars), even a tiny, unimportant price gap would still give a p-value close to 0. To know the actual size, I need to look at the confidence interval and the effect size, not the p-value.

## 3. Confidence interval

In [188]:
diff = auto.mean() - manual.mean()
se = np.sqrt(auto.var() / len(auto) + manual.var() / len(manual))
ci_low, ci_high = diff - 1.96 * se, diff + 1.96 * se

In [189]:
print(f"Mean price difference (Automatic - Manual): {diff:.0f} AZN")
print(f"95% CI: [{ci_low:.0f}, {ci_high:.0f}] AZN")

Mean price difference (Automatic - Manual): 19456 AZN
95% CI: [19227, 19684] AZN


Using this method, I'd expect 95% of intervals built the same way to catch the true average price gap between automatic and manual cars. My best guess for that gap is about 19,456 AZN, and the interval [19,227, 19,684] AZN is pretty narrow, which means the estimate is fairly precise. The interval doesn't include 0, which matches the p-value being basically 0: both are pointing to a real difference. But the interval gives me something the p-value can't, it tells me the difference isn't just "there," it's specifically around 19,000–20,000 AZN. That's a large gap in real money for this market.

## 4. Statistical significance vs. practical significance

With a big enough sample, almost any tiny, meaningless difference will come out "statistically significant," because p-values shrink as the sample grows, even if the actual size of the gap never changes. My sample here has over 135,000 cars, so I can't just trust the p-value to tell me if the result actually matters.

In [190]:
pooled_std = np.sqrt(((len(auto) - 1) * auto.var() + (len(manual) - 1) * manual.var()) / (len(auto) + len(manual) - 2))
cohens_d = (auto.mean() - manual.mean()) / pooled_std

In [191]:
print(f"Cohen's d: {cohens_d:.3f}")
print(f"Automatic is about {diff / manual.mean() * 100:.0f}% more expensive on average than manual")

Cohen's d: 0.664
Automatic is about 184% more expensive on average than manual


## 5. Interpreting the ANOVA, Bonferroni results

In [192]:
top5_brands = ["Mercedes", "Hyundai", "Kia", "Toyota", "LADA (VAZ)"]
groups = [df_clean.loc[df_clean["Marka"] == b, "price_azn"].values for b in top5_brands]


In [193]:
F_stat, p_value_anova = stats.f_oneway(*groups)
grand_mean = np.concatenate(groups).mean()
ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
ss_total = sum(((g - grand_mean) ** 2).sum() for g in groups)
eta_squared = ss_between / ss_total

In [194]:
print(f"ANOVA F-statistic: {F_stat:.2f}")
print(f"p-value: {p_value_anova:.2e}")
print(f"Eta-squared: {eta_squared:.3f}")

ANOVA F-statistic: 1942.50
p-value: 0.00e+00
Eta-squared: 0.082


p = 0 only means that somewhere among the 5 brands, at least one has a different average price. It doesn't say which brand, how many brands, or by how much. Eta-squared = 0.082 tells me brand only explains about 8% of the total price variation, so most of what drives price comes from other things, not brand by itself.

In [195]:
alpha = 0.05
n_comparisons = len(list(combinations(top5_brands, 2)))
alpha_corrected = alpha / n_comparisons


In [196]:
posthoc_results = []
for b1, b2 in combinations(top5_brands, 2):
    g1 = df_clean.loc[df_clean["Marka"] == b1, "price_azn"]
    g2 = df_clean.loc[df_clean["Marka"] == b2, "price_azn"]
    t_stat, p_raw = stats.ttest_ind(g1, g2, equal_var=False)
    diff2 = g1.mean() - g2.mean()
    se2 = np.sqrt(g1.var() / len(g1) + g2.var() / len(g2))
    ci_l, ci_h = diff2 - 1.96 * se2, diff2 + 1.96 * se2
    posthoc_results.append({
        "Pair": f"{b1} vs {b2}",
        "Mean diff": round(diff2, 0),
        "95% CI": f"[{ci_l:.0f}, {ci_h:.0f}]",
        "p-value (raw)": p_raw,
        "Significant (Bonferroni alpha=0.005)": p_raw < alpha_corrected,
    })

In [197]:
posthoc_df = pd.DataFrame(posthoc_results)
posthoc_df

,Pair,Mean diff,95% CI,p-value (raw),Significant (Bonferroni alpha=0.005)
0,Mercedes vs Hyundai,1786.0,"[1303, 2269]",4.516971e-13,True
1,Mercedes vs Kia,-115.0,"[-616, 386]",6.527356e-01,False
2,Mercedes vs Toyota,-4638.0,"[-5231, -4044]",9.513534e-53,True
3,Mercedes vs LADA (VAZ),18386.0,"[17915, 18857]",0.000000e+00,True
4,Hyundai vs Kia,-1901.0,"[-2136, -1667]",1.361007e-56,True
5,Hyundai vs Toyota,-6424.0,"[-6819, -6028]",3.022574e-216,True
6,Hyundai vs LADA (VAZ),16600.0,"[16440, 16760]",0.000000e+00,True
7,Kia vs Toyota,-4523.0,"[-4940, -4106]",3.173328e-99,True
8,Kia vs LADA (VAZ),18501.0,"[18294, 18708]",0.000000e+00,True
9,Toyota vs LADA (VAZ),23024.0,"[22644, 23404]",0.000000e+00,True


p = 0.65 there, and the 95% CI for that pair includes 0. That does not prove Mercedes and Kia are priced the same, it just means this sample doesn't give me enough evidence to say they're different. Not finding proof of a difference isn't the same as proving there isn't one.every other pair has a CI that skips over 0 and stays significant even after the stricter Bonferroni cutoff (α = 0.005), so I trust those 9 brand comparisons, including which direction the gap goes and roughly how big it is, which the CI gives me directly.

## 6. Interpreting the chi-square result (gearbox vs. drivetrain)

In [198]:
contingency_table = pd.crosstab(df_clean["Sürətlər qutusu"], df_clean["Ötürücü"])
chi2_stat, p_chi2, dof_chi2, expected_freq = stats.chi2_contingency(contingency_table)


In [199]:
n_total = contingency_table.sum().sum()
cramers_v = np.sqrt(chi2_stat / (n_total * (min(contingency_table.shape) - 1)))

In [200]:
print(f"Chi-square statistic: {chi2_stat:.1f}")
print(f"p-value: {p_chi2:.2e}")
print(f"Cramer's V: {cramers_v:.3f}")

Chi-square statistic: 14246.6
p-value: 0.00e+00
Cramer's V: 0.218


p = 0 means gearbox type and drivetrain type are connected in this data, knowing one gives me some information about the other, and this connection is very unlikely to just be random noise given how big the sample is. But Cramer's V = 0.218 says that connection is only weak-to-moderate. If I only looked at "p < 0.05, significant," I might wrongly think gearbox almost decides drivetrain.

## 8. Final

Gearbox vs. price: p = 0 says the gap is real. CI [19,227, 19,684] AZN and Cohen's d = 0.66 say it's also big. Statistical and real-world results agree here.

Brand vs. price (ANOVA): p = 0 says at least one brand differs, but eta-squared = 0.082 says brand only explains a small part of the price gap. 9 of 10 brand pairs are confirmed different; Mercedes vs. Kia is the one I can't call, not because they're equal, just not enough evidence.

Gearbox vs. drivetrain (chi-square): p = 0 says they're connected, but Cramer's V = 0.218 says only weakly.

Pattern across all three: p-value only answers "is this probably not noise?" How big or how much it matters always came from the CI and effect size.